# EvalWeaver Persona Market Simulator v2

This notebook simulates launch dynamics for the EvalWeaver marketplace.

It fixes two issues from the prior version:

1. Robust LLM panel parsing:
   - handles JSON returned as `{"responses": [...]}`,
   - handles raw `[...]` list responses,
   - handles dicts keyed by persona id,
   - falls back persona-by-persona if a batch is malformed.

2. Better sampling frame:
   - no blind random sampling of everyone in Singapore/US,
   - uses a targeted-but-not-too-narrow launch audience:
     founders, marketers, creators, sales/BD, consultants/agencies, researchers/writers, students/job seekers, and skeptical controls.
   - deterministic quotas, so runs are reproducible and easier to debug.

The goal is not to predict the real market. It stress-tests assumptions about pricing, scorer positioning, trust, and creator marketplace dynamics.

## Cell 0 — Config

Set `USE_LLM_PERSONA_PANEL=True` only when Colab AI / Gemini is available.

Default mode runs fully deterministically and produces outputs immediately.

In [2]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple
import json, re, math, random, os, uuid, traceback

import numpy as np
import pandas as pd

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, HTML
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    def display(x): print(x)
    def Markdown(x): return x
    def HTML(x): return x

SEED = 20260608
random.seed(SEED)
np.random.seed(SEED)

OUTDIR = Path("/content/evalweaver_persona_sim_v2_outputs") if Path("/content").exists() else Path("/mnt/data/evalweaver_persona_sim_v2_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)


# Preferred Colab model selection.
# If Colab exposes Gemini 3.5 Flash, choose it; otherwise fall back to another Flash model.
MODEL_NAME_OVERRIDE = None  # e.g. "models/gemini-3.5-flash-preview" if you want to hardcode exact Colab listing.
MODEL_PREFERENCE_SUBSTRINGS = ["3.5", "flash"]

# Toggle LLM persona panel.
USE_LLM_PERSONA_PANEL = True
LLM_BATCH_SIZE = 10

# Markets to simulate.
MARKETS = ["Singapore", "United States"]

# Price points for pay-to-reveal.
PRICE_POINTS = [1.99, 2.99, 4.99, 7.99, 9.99, 19.99]

# Demo Stripe-ish assumptions; edit as needed.
PAYMENT_FEES = {
    "Singapore": {"rate": 0.034, "fixed": 0.50, "currency": "SGD"},
    "United States": {"rate": 0.029, "fixed": 0.30, "currency": "USD"},
}

# Model/API estimated cost placeholders per reveal.
ESTIMATED_API_COST = {
    "preset_reveal": 0.35,
    "custom_evaluator_build": 4.50,
}

# Persona count target. This is deterministic quota construction, not random citizen sampling.
PERSONAS_PER_MARKET = 100

print("✅ Config loaded")
print("Markets:", MARKETS)
print("Personas per market:", PERSONAS_PER_MARKET)
print("LLM panel:", USE_LLM_PERSONA_PANEL)
print("Output dir:", OUTDIR)

✅ Config loaded
Markets: ['Singapore', 'United States']
Personas per market: 100
LLM panel: True
Output dir: /content/evalweaver_persona_sim_v2_outputs


## Cell 1 — Robust JSON extraction and LLM call wrapper

This directly fixes the error:

```text
AttributeError("'list' object has no attribute 'get'")
```

The parser now normalizes dict/list/keyed-dict outputs into a consistent list of responses.

In [3]:
RUN_LEDGER = []

def approx_tokens(text: str) -> int:
    return max(1, int(len(str(text).split()) * 1.35))

def extract_json_from_text(text: str):
    text = str(text).strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    if not text.startswith("{") and not text.startswith("["):
        starts = [i for i in [text.find("{"), text.find("[")] if i >= 0]
        if starts:
            start = min(starts)
            end = max(text.rfind("}"), text.rfind("]"))
            if end > start:
                text = text[start:end+1]
    return json.loads(text)

def normalize_panel_output(parsed: Any, expected_persona_ids: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    """
    Accepts:
      {"responses": [...]}
      {"results": [...]}
      {"persona_responses": [...]}
      [...]
      {"persona_id_1": {...}, "persona_id_2": {...}}
    Returns list[dict].
    """
    if parsed is None:
        return []

    if isinstance(parsed, list):
        responses = parsed

    elif isinstance(parsed, dict):
        for key in ["responses", "results", "persona_responses", "panel", "data"]:
            if isinstance(parsed.get(key), list):
                responses = parsed[key]
                break
        else:
            # Maybe dict keyed by persona_id.
            dict_values = []
            for k, v in parsed.items():
                if isinstance(v, dict):
                    item = {"persona_id": k, **v}
                    dict_values.append(item)
            if dict_values:
                responses = dict_values
            else:
                responses = [parsed]

    else:
        return []

    normalized = []
    for i, r in enumerate(responses):
        if not isinstance(r, dict):
            continue
        r = dict(r)
        if "persona_id" not in r and expected_persona_ids and i < len(expected_persona_ids):
            r["persona_id"] = expected_persona_ids[i]
        normalized.append(r)

    return normalized

def safe_extract_panel_json(raw: str, expected_persona_ids: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    try:
        parsed = extract_json_from_text(raw)
        return normalize_panel_output(parsed, expected_persona_ids=expected_persona_ids)
    except Exception as e:
        print("⚠️ Panel JSON parse failed:", repr(e))
        print("Raw preview:")
        print(str(raw)[:1500])
        return []

def available_colab_ai():
    try:
        from google.colab import ai  # type: ignore
        return ai
    except Exception:
        return None

def choose_model():
    ai = available_colab_ai()
    if ai is None:
        return None

    models = list(ai.list_models())

    # Manual override wins if set and present.
    if MODEL_NAME_OVERRIDE:
        for m in models:
            if str(m) == MODEL_NAME_OVERRIDE or MODEL_NAME_OVERRIDE in str(m):
                return m
        print("⚠️ MODEL_NAME_OVERRIDE not found in ai.list_models(); falling back to preference search.")

    # Prefer Gemini 3.5 Flash if Colab exposes it.
    preferred = []
    for m in models:
        s = str(m).lower()
        if all(substr.lower() in s for substr in MODEL_PREFERENCE_SUBSTRINGS):
            preferred.append(m)
    if preferred:
        return preferred[0]

    # Otherwise prefer any Flash model.
    for m in models:
        if "flash" in str(m).lower():
            return m

    return models[0] if models else None

def call_llm_text(prompt: str, agent_name: str = "PersonaPanelAgent") -> str:
    ai = available_colab_ai()
    if ai is None:
        raise RuntimeError("google.colab.ai is not available. Set USE_LLM_PERSONA_PANEL=False.")
    model_name = choose_model()
    chunks = []
    for chunk in ai.generate_text(prompt=prompt, model_name=model_name, stream=False):
        if chunk is not None:
            chunks.append(chunk)
    raw = "".join(chunks)
    RUN_LEDGER.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "agent_name": agent_name,
        "model_name": str(model_name),
        "llm_calls": 1,
        "input_tokens_est": approx_tokens(prompt),
        "output_tokens_est": approx_tokens(raw),
        "status": "ok",
    })
    return raw

def show_ledger():
    display(pd.DataFrame(RUN_LEDGER) if RUN_LEDGER else pd.DataFrame([{"status": "no calls yet"}]))

print("✅ Robust parser loaded")

✅ Robust parser loaded


## Cell 2 — Define the launch sampling frame

Instead of sampling everyone, this builds a deterministic launch panel from people plausibly likely to:

- post online,
- write for work,
- buy a “make this more ___” tool,
- build or sell custom scorer/improver functions,
- or provide useful skeptical controls.

This avoids overfitting to only AI founders while still avoiding a generic population sample.

In [4]:
SEGMENT_FRAME = [
    {
        "segment": "sme_owner_operator",
        "weight": 0.16,
        "why_relevant": "Writes sales posts, customer replies, hiring posts, and product/service pages.",
        "likely_jobs": ["SME owner", "shop owner", "clinic manager", "services business owner", "e-commerce operator"],
        "content_jobs": ["sales copy", "customer updates", "offer posts", "website copy"],
    },
    {
        "segment": "startup_founder_operator",
        "weight": 0.14,
        "why_relevant": "Needs pitch copy, launch posts, investor updates, product positioning.",
        "likely_jobs": ["startup founder", "product operator", "early employee", "solo SaaS builder"],
        "content_jobs": ["launch post", "investor update", "landing page", "demo script"],
    },
    {
        "segment": "marketing_growth_lead",
        "weight": 0.14,
        "why_relevant": "Direct buyer/user for subjective content improvement and A/B testing.",
        "likely_jobs": ["growth marketer", "content marketer", "brand lead", "social media manager"],
        "content_jobs": ["ads", "landing pages", "email campaigns", "social posts"],
    },
    {
        "segment": "creator_coach_consultant",
        "weight": 0.13,
        "why_relevant": "Can use and monetize a personal taste/improver scorer with followers.",
        "likely_jobs": ["LinkedIn creator", "coach", "consultant", "newsletter writer", "course creator"],
        "content_jobs": ["thought leadership", "newsletter", "sales page", "coaching offer"],
    },
    {
        "segment": "sales_bd_customer_success",
        "weight": 0.12,
        "why_relevant": "Writes outreach, follow-ups, proposal text, customer comms.",
        "likely_jobs": ["sales manager", "BD executive", "customer success lead", "account manager"],
        "content_jobs": ["outreach", "follow-up", "proposal", "customer message"],
    },
    {
        "segment": "agency_freelancer",
        "weight": 0.10,
        "why_relevant": "May buy custom evaluators for client workflows or publish scorers.",
        "likely_jobs": ["copywriter", "SEO freelancer", "creative agency lead", "AI automation agency"],
        "content_jobs": ["client copy", "campaign variants", "brand voice", "lead magnets"],
    },
    {
        "segment": "researcher_technical_writer",
        "weight": 0.08,
        "why_relevant": "Adjacent use for scientific/readable/credible writing, not just persuasive.",
        "likely_jobs": ["researcher", "technical writer", "data scientist", "PhD student"],
        "content_jobs": ["abstract", "technical blog", "grant text", "research summary"],
    },
    {
        "segment": "student_job_seeker",
        "weight": 0.07,
        "why_relevant": "Lower willingness-to-pay but high need for cover letters, profiles, applications.",
        "likely_jobs": ["student", "job seeker", "intern", "early-career professional"],
        "content_jobs": ["resume bullets", "cover letter", "LinkedIn post", "application essay"],
    },
    {
        "segment": "skeptical_control",
        "weight": 0.06,
        "why_relevant": "Useful control group: writes rarely or distrusts AI/pay-to-reveal.",
        "likely_jobs": ["operations manager", "teacher", "finance analyst", "public sector administrator"],
        "content_jobs": ["internal memo", "occasional post", "formal email"],
    },
]

MARKET_CONTEXTS = {
    "Singapore": {
        "currency": "SGD",
        "locations": ["CBD", "Tanjong Pagar", "Jurong East", "Tampines", "Paya Lebar", "Woodlands", "One-North", "Novena"],
        "channels": ["LinkedIn", "WhatsApp Business", "Instagram", "email", "website"],
        "local_hooks": ["SME productivity", "grant applications", "regional expansion", "customer support quality", "lean teams"],
    },
    "United States": {
        "currency": "USD",
        "locations": ["San Francisco", "New York", "Austin", "Seattle", "Miami", "Chicago", "Boston", "Atlanta"],
        "channels": ["LinkedIn", "X/Twitter", "email", "website", "newsletter"],
        "local_hooks": ["startup launch", "creator monetization", "sales conversion", "agency workflows", "SaaS growth"],
    },
}

def deterministic_quota_counts(total: int, frame: List[Dict[str, Any]]) -> Dict[str, int]:
    raw = {x["segment"]: total * x["weight"] for x in frame}
    counts = {k: int(math.floor(v)) for k, v in raw.items()}
    remaining = total - sum(counts.values())
    remainders = sorted([(v - math.floor(v), k) for k, v in raw.items()], reverse=True)
    for _, k in remainders[:remaining]:
        counts[k] += 1
    return counts

def build_persona_panel_for_market(market: str, total: int = PERSONAS_PER_MARKET) -> List[Dict[str, Any]]:
    ctx = MARKET_CONTEXTS[market]
    counts = deterministic_quota_counts(total, SEGMENT_FRAME)
    personas = []
    idx = 0
    for seg in SEGMENT_FRAME:
        n = counts[seg["segment"]]
        for j in range(n):
            job = seg["likely_jobs"][j % len(seg["likely_jobs"])]
            content_job = seg["content_jobs"][(j // len(seg["likely_jobs"])) % len(seg["content_jobs"])]
            location = ctx["locations"][(j + len(personas)) % len(ctx["locations"])]
            channel = ctx["channels"][(j * 2 + len(personas)) % len(ctx["channels"])]
            hook = ctx["local_hooks"][(j * 3 + len(personas)) % len(ctx["local_hooks"])]
            idx += 1
            personas.append({
                "persona_id": f"{market[:2].upper()}_{idx:03d}",
                "market": market,
                "currency": ctx["currency"],
                "segment": seg["segment"],
                "job_role": job,
                "location_context": location,
                "primary_channel": channel,
                "content_job_to_be_done": content_job,
                "local_hook": hook,
                "why_relevant": seg["why_relevant"],
                "ai_comfort": ["low", "medium", "high"][(j + idx) % 3],
                "budget_sensitivity": ["high", "medium", "low"][(j + 2*idx) % 3],
                "posting_frequency": ["rare", "monthly", "weekly", "daily"][(j + idx) % 4],
            })
    return personas

personas = []
for market in MARKETS:
    personas.extend(build_persona_panel_for_market(market, PERSONAS_PER_MARKET))

personas_df = pd.DataFrame(personas)

checks = {
    "targeted_not_general_population": set(personas_df["segment"]) == {x["segment"] for x in SEGMENT_FRAME},
    "both_markets_present": set(personas_df["market"]) == set(MARKETS),
    "deterministic_size": len(personas_df) == PERSONAS_PER_MARKET * len(MARKETS),
    "has_skeptical_control": "skeptical_control" in set(personas_df["segment"]),
}
print("Checks:", checks)
display(personas_df.head(12))
display(personas_df.groupby(["market", "segment"]).size().reset_index(name="count"))

Checks: {'targeted_not_general_population': True, 'both_markets_present': True, 'deterministic_size': True, 'has_skeptical_control': True}


,persona_id,market,currency,segment,job_role,location_context,primary_channel,content_job_to_be_done,local_hook,why_relevant,ai_comfort,budget_sensitivity,posting_frequency
0,SI_001,Singapore,SGD,sme_owner_operator,SME owner,CBD,LinkedIn,sales copy,SME productivity,"Writes sales posts, customer replies, hiring p...",medium,low,monthly
1,SI_002,Singapore,SGD,sme_owner_operator,shop owner,Jurong East,email,sales copy,lean teams,"Writes sales posts, customer replies, hiring p...",low,low,daily
2,SI_003,Singapore,SGD,sme_owner_operator,clinic manager,Paya Lebar,WhatsApp Business,sales copy,customer support quality,"Writes sales posts, customer replies, hiring p...",high,low,monthly
3,SI_004,Singapore,SGD,sme_owner_operator,services business owner,One-North,website,sales copy,regional expansion,"Writes sales posts, customer replies, hiring p...",medium,low,daily
4,SI_005,Singapore,SGD,sme_owner_operator,e-commerce operator,CBD,Instagram,sales copy,grant applications,"Writes sales posts, customer replies, hiring p...",low,low,monthly
5,SI_006,Singapore,SGD,sme_owner_operator,SME owner,Jurong East,LinkedIn,customer updates,SME productivity,"Writes sales posts, customer replies, hiring p...",high,low,daily
6,SI_007,Singapore,SGD,sme_owner_operator,shop owner,Paya Lebar,email,customer updates,lean teams,"Writes sales posts, customer replies, hiring p...",medium,low,monthly
7,SI_008,Singapore,SGD,sme_owner_operator,clinic manager,One-North,WhatsApp Business,customer updates,customer support quality,"Writes sales posts, customer replies, hiring p...",low,low,daily
8,SI_009,Singapore,SGD,sme_owner_operator,services business owner,CBD,website,customer updates,regional expansion,"Writes sales posts, customer replies, hiring p...",high,low,monthly
9,SI_010,Singapore,SGD,sme_owner_operator,e-commerce operator,Jurong East,Instagram,customer updates,grant applications,"Writes sales posts, customer replies, hiring p...",medium,low,daily


,market,segment,count
0,Singapore,agency_freelancer,10
1,Singapore,creator_coach_consultant,13
2,Singapore,marketing_growth_lead,14
3,Singapore,researcher_technical_writer,8
4,Singapore,sales_bd_customer_success,12
5,Singapore,skeptical_control,6
6,Singapore,sme_owner_operator,16
7,Singapore,startup_founder_operator,14
8,Singapore,student_job_seeker,7
9,United States,agency_freelancer,10


## Cell 3 — Scorer marketplace cards

These are the products personas choose among. Keep this small for the hackathon.

In [5]:
SCORER_CARDS = [
    {
        "scorer_id": "persuasive_without_hype",
        "title": "Persuasive Without Hype",
        "dynamic_variable": "persuasive",
        "creator": "EvalWeaver Seed",
        "positioning": "Makes copy clearer, more credible, and more action-oriented without sounding fake.",
        "best_for": ["sales copy", "landing page", "launch post", "proposal"],
        "price_to_reveal": 4.99,
        "custom_build_price": 39.00,
        "avg_rating_demo": 4.7,
        "paid_uses_demo": 128,
    },
    {
        "scorer_id": "linkedin_creator_hook",
        "title": "LinkedIn Creator Hook",
        "dynamic_variable": "viral",
        "creator": "Creator A",
        "positioning": "Improves hook, story tension, and comment-worthy ending.",
        "best_for": ["LinkedIn post", "newsletter", "thought leadership"],
        "price_to_reveal": 6.99,
        "custom_build_price": 79.00,
        "avg_rating_demo": 4.5,
        "paid_uses_demo": 42,
    },
    {
        "scorer_id": "scientific_but_readable",
        "title": "Scientific But Readable",
        "dynamic_variable": "scientific",
        "creator": "EvalWeaver Seed",
        "positioning": "Makes claims more evidence-grounded, careful, and easier to read.",
        "best_for": ["research summary", "technical blog", "grant text"],
        "price_to_reveal": 4.99,
        "custom_build_price": 59.00,
        "avg_rating_demo": 4.8,
        "paid_uses_demo": 73,
    },
    {
        "scorer_id": "investor_ready",
        "title": "Investor-Ready",
        "dynamic_variable": "investor-ready",
        "creator": "Creator B",
        "positioning": "Tightens business claims, traction narrative, and investor logic.",
        "best_for": ["pitch", "investor update", "demo script"],
        "price_to_reveal": 9.99,
        "custom_build_price": 99.00,
        "avg_rating_demo": 4.4,
        "paid_uses_demo": 31,
    },
]

display(pd.DataFrame(SCORER_CARDS))

,scorer_id,title,dynamic_variable,creator,positioning,best_for,price_to_reveal,custom_build_price,avg_rating_demo,paid_uses_demo
0,persuasive_without_hype,Persuasive Without Hype,persuasive,EvalWeaver Seed,"Makes copy clearer, more credible, and more ac...","[sales copy, landing page, launch post, proposal]",4.99,39.0,4.7,128
1,linkedin_creator_hook,LinkedIn Creator Hook,viral,Creator A,"Improves hook, story tension, and comment-wort...","[LinkedIn post, newsletter, thought leadership]",6.99,79.0,4.5,42
2,scientific_but_readable,Scientific But Readable,scientific,EvalWeaver Seed,"Makes claims more evidence-grounded, careful, ...","[research summary, technical blog, grant text]",4.99,59.0,4.8,73
3,investor_ready,Investor-Ready,investor-ready,Creator B,"Tightens business claims, traction narrative, ...","[pitch, investor update, demo script]",9.99,99.0,4.4,31


## Cell 4 — Deterministic persona decision model

This is the fallback simulator and also the sanity-check baseline for LLM persona panels.

It asks each persona the implicit questions:

1. Do they have a strong use case?
2. Which scorer fits their job-to-be-done?
3. Does the teaser create enough trust?
4. Are they willing to pay this reveal price?
5. Would they build a custom evaluator?
6. Would they publish a scorer to earn revenue share?
7. Why would they bounce?

In [6]:
SEGMENT_BASE_INTENT = {
    "sme_owner_operator": 0.58,
    "startup_founder_operator": 0.72,
    "marketing_growth_lead": 0.78,
    "creator_coach_consultant": 0.75,
    "sales_bd_customer_success": 0.65,
    "agency_freelancer": 0.70,
    "researcher_technical_writer": 0.52,
    "student_job_seeker": 0.42,
    "skeptical_control": 0.18,
}

SEGMENT_CUSTOM_BUILD = {
    "sme_owner_operator": 0.22,
    "startup_founder_operator": 0.42,
    "marketing_growth_lead": 0.45,
    "creator_coach_consultant": 0.50,
    "sales_bd_customer_success": 0.26,
    "agency_freelancer": 0.55,
    "researcher_technical_writer": 0.28,
    "student_job_seeker": 0.08,
    "skeptical_control": 0.04,
}

SEGMENT_CREATOR_PUBLISH = {
    "sme_owner_operator": 0.08,
    "startup_founder_operator": 0.18,
    "marketing_growth_lead": 0.22,
    "creator_coach_consultant": 0.62,
    "sales_bd_customer_success": 0.10,
    "agency_freelancer": 0.52,
    "researcher_technical_writer": 0.18,
    "student_job_seeker": 0.12,
    "skeptical_control": 0.02,
}

def scorer_fit(persona: Dict[str, Any], card: Dict[str, Any]) -> float:
    seg = persona["segment"]
    content = persona["content_job_to_be_done"].lower()
    title = card["title"].lower()
    dyn = card["dynamic_variable"].lower()

    score = 0.25

    if card["scorer_id"] == "persuasive_without_hype":
        if seg in {"sme_owner_operator","startup_founder_operator","marketing_growth_lead","sales_bd_customer_success","agency_freelancer"}:
            score += 0.35
        if any(k in content for k in ["sales", "landing", "offer", "proposal", "launch"]):
            score += 0.20

    if card["scorer_id"] == "linkedin_creator_hook":
        if seg in {"creator_coach_consultant", "marketing_growth_lead", "agency_freelancer"}:
            score += 0.35
        if any(k in content for k in ["post", "newsletter", "thought"]):
            score += 0.25

    if card["scorer_id"] == "scientific_but_readable":
        if seg in {"researcher_technical_writer","student_job_seeker","startup_founder_operator"}:
            score += 0.30
        if any(k in content for k in ["technical", "grant", "research", "abstract"]):
            score += 0.30

    if card["scorer_id"] == "investor_ready":
        if seg in {"startup_founder_operator", "agency_freelancer", "sme_owner_operator"}:
            score += 0.30
        if any(k in content for k in ["investor", "pitch", "demo", "update"]):
            score += 0.35

    if persona["ai_comfort"] == "high":
        score += 0.06
    elif persona["ai_comfort"] == "low":
        score -= 0.08

    return max(0.01, min(score, 0.98))

def price_sensitivity_multiplier(persona: Dict[str, Any], price: float) -> float:
    sensitivity = persona["budget_sensitivity"]
    market = persona["market"]
    # rough reference prices in local currency
    if sensitivity == "low":
        reference = 12.0 if market == "Singapore" else 12.0
    elif sensitivity == "medium":
        reference = 6.0
    else:
        reference = 3.0
    return 1 / (1 + math.exp((price - reference) / max(1.0, reference * 0.35)))

def trust_multiplier(persona: Dict[str, Any], card: Dict[str, Any]) -> float:
    trust = 0.75
    if card["paid_uses_demo"] > 50:
        trust += 0.08
    if card["avg_rating_demo"] >= 4.6:
        trust += 0.08
    if persona["ai_comfort"] == "low":
        trust -= 0.12
    if persona["segment"] == "skeptical_control":
        trust -= 0.18
    return max(0.1, min(1.1, trust))

def choose_best_scorer(persona: Dict[str, Any], cards: List[Dict[str, Any]]) -> Tuple[Dict[str, Any], float]:
    fits = [(scorer_fit(persona, c), c) for c in cards]
    fits.sort(key=lambda x: x[0], reverse=True)
    return fits[0][1], fits[0][0]

def deterministic_decision(persona: Dict[str, Any], price_override: Optional[float] = None) -> Dict[str, Any]:
    card, fit = choose_best_scorer(persona, SCORER_CARDS)
    price = price_override if price_override is not None else card["price_to_reveal"]

    base = SEGMENT_BASE_INTENT[persona["segment"]]
    price_mult = price_sensitivity_multiplier(persona, price)
    trust = trust_multiplier(persona, card)
    freq_bonus = {"rare": -0.06, "monthly": 0.0, "weekly": 0.07, "daily": 0.12}[persona["posting_frequency"]]

    try_prob = max(0.01, min(0.98, base * 0.65 + fit * 0.35 + freq_bonus))
    reveal_prob = max(0.01, min(0.98, try_prob * price_mult * trust))
    custom_prob = max(0.0, min(0.90, SEGMENT_CUSTOM_BUILD[persona["segment"]] * trust * (1.05 if persona["ai_comfort"] == "high" else 0.85)))
    publish_prob = max(0.0, min(0.90, SEGMENT_CREATOR_PUBLISH[persona["segment"]] * (1.1 if card["creator"] != "EvalWeaver Seed" else 0.95)))

    # Deterministic pseudo-random draw from persona_id and price.
    h = abs(hash((persona["persona_id"], card["scorer_id"], round(price, 2), SEED))) % 10000 / 10000
    paid_reveal = h < reveal_prob
    custom_build = (abs(hash(("custom", persona["persona_id"], SEED))) % 10000 / 10000) < custom_prob
    publish_scorer = (abs(hash(("publish", persona["persona_id"], SEED))) % 10000 / 10000) < publish_prob

    if reveal_prob < 0.20:
        bounce_reason = "price/trust too weak"
    elif fit < 0.45:
        bounce_reason = "scorer not relevant enough"
    elif persona["ai_comfort"] == "low":
        bounce_reason = "AI trust concern"
    else:
        bounce_reason = "may need stronger teaser or proof"

    return {
        "persona_id": persona["persona_id"],
        "market": persona["market"],
        "segment": persona["segment"],
        "job_role": persona["job_role"],
        "content_job_to_be_done": persona["content_job_to_be_done"],
        "selected_scorer_id": card["scorer_id"],
        "selected_scorer_title": card["title"],
        "price": price,
        "try_probability": round(try_prob, 3),
        "reveal_probability": round(reveal_prob, 3),
        "custom_build_probability": round(custom_prob, 3),
        "publish_scorer_probability": round(publish_prob, 3),
        "paid_reveal": bool(paid_reveal),
        "custom_build": bool(custom_build),
        "publish_scorer": bool(publish_scorer),
        "trust_reason": f"fit={fit:.2f}; channel={persona['primary_channel']}; use_case={persona['content_job_to_be_done']}",
        "bounce_reason": bounce_reason,
    }

deterministic_rows = [deterministic_decision(p) for p in personas]
deterministic_df = pd.DataFrame(deterministic_rows)

display(deterministic_df.head(12))
display(deterministic_df.groupby(["market", "segment"]).agg(
    personas=("persona_id", "count"),
    avg_reveal_prob=("reveal_probability", "mean"),
    paid_reveals=("paid_reveal", "sum"),
    custom_builds=("custom_build", "sum"),
    publish_scorers=("publish_scorer", "sum"),
).reset_index())

,persona_id,market,segment,job_role,content_job_to_be_done,selected_scorer_id,selected_scorer_title,price,try_probability,reveal_probability,custom_build_probability,publish_scorer_probability,paid_reveal,custom_build,publish_scorer,trust_reason,bounce_reason
0,SI_001,Singapore,sme_owner_operator,SME owner,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.657,0.503,0.170,0.076,False,False,False,fit=0.80; channel=LinkedIn; use_case=sales copy,may need stronger teaser or proof
1,SI_002,Singapore,sme_owner_operator,shop owner,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.749,0.498,0.148,0.076,True,False,False,fit=0.72; channel=email; use_case=sales copy,AI trust concern
2,SI_003,Singapore,sme_owner_operator,clinic manager,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.678,0.519,0.210,0.076,True,False,False,fit=0.86; channel=WhatsApp Business; use_case=...,may need stronger teaser or proof
3,SI_004,Singapore,sme_owner_operator,services business owner,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.777,0.595,0.170,0.076,True,False,False,fit=0.80; channel=website; use_case=sales copy,may need stronger teaser or proof
4,SI_005,Singapore,sme_owner_operator,e-commerce operator,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.629,0.418,0.148,0.076,False,False,False,fit=0.72; channel=Instagram; use_case=sales copy,AI trust concern
5,SI_006,Singapore,sme_owner_operator,SME owner,customer updates,investor_ready,Investor-Ready,9.99,0.833,0.386,0.173,0.088,False,False,False,fit=0.96; channel=LinkedIn; use_case=customer ...,may need stronger teaser or proof
6,SI_007,Singapore,sme_owner_operator,shop owner,customer updates,investor_ready,Investor-Ready,9.99,0.692,0.320,0.140,0.088,False,True,False,fit=0.90; channel=email; use_case=customer upd...,may need stronger teaser or proof
7,SI_008,Singapore,sme_owner_operator,clinic manager,customer updates,investor_ready,Investor-Ready,9.99,0.784,0.305,0.118,0.088,False,False,True,fit=0.82; channel=WhatsApp Business; use_case=...,AI trust concern
8,SI_009,Singapore,sme_owner_operator,services business owner,customer updates,investor_ready,Investor-Ready,9.99,0.713,0.330,0.173,0.088,False,True,False,fit=0.96; channel=website; use_case=customer u...,may need stronger teaser or proof
9,SI_010,Singapore,sme_owner_operator,e-commerce operator,customer updates,investor_ready,Investor-Ready,9.99,0.812,0.376,0.140,0.088,False,False,False,fit=0.90; channel=Instagram; use_case=customer...,may need stronger teaser or proof


,market,segment,personas,avg_reveal_prob,paid_reveals,custom_builds,publish_scorers
0,Singapore,agency_freelancer,10,0.492700,3,3,5
1,Singapore,creator_coach_consultant,13,0.012538,0,2,7
2,Singapore,marketing_growth_lead,14,0.566000,10,5,5
3,Singapore,researcher_technical_writer,8,0.349375,3,0,1
4,Singapore,sales_bd_customer_success,12,0.483083,7,2,1
5,Singapore,skeptical_control,6,0.099667,0,0,0
6,Singapore,sme_owner_operator,16,0.458750,7,2,1
7,Singapore,startup_founder_operator,14,0.281071,3,3,3
8,Singapore,student_job_seeker,7,0.338714,4,0,3
9,United States,agency_freelancer,10,0.492700,4,2,4


## Cell 5 — Optional LLM PersonaPanelAgent

This asks personas in batches, not one-by-one.

Estimated calls:

```python
ceil(num_personas / LLM_BATCH_SIZE)
```

For 144 personas and batch size 12, that is 12 calls total.

If a batch fails or returns malformed JSON, the notebook falls back to deterministic decisions for that batch.

In [7]:
def build_persona_panel_prompt(batch: List[Dict[str, Any]]) -> str:
    return f"""
Return JSON only. No markdown. No prose.

You are PersonaPanelAgent. Simulate likely market reactions for EvalWeaver.

Product:
EvalWeaver lets users choose "make this more ____", paste text, see a score-lift teaser, then pay to reveal improved outputs. Creators can publish custom scorer/improvers and earn revenue share.

Scorer cards:
{json.dumps(SCORER_CARDS, indent=2)}

Personas:
{json.dumps(batch, indent=2)}

For each persona, answer:
1. Which scorer would they try first?
2. Would they pay to reveal at the listed price?
3. Would they consider building a custom evaluator?
4. Would they publish a scorer/improver to earn revenue share?
5. What is the main trust reason?
6. What is the main bounce reason?

Return exactly:
{{
  "responses": [
    {{
      "persona_id": "...",
      "selected_scorer_id": "...",
      "try_probability": 0.0,
      "reveal_probability": 0.0,
      "custom_build_probability": 0.0,
      "publish_scorer_probability": 0.0,
      "paid_reveal": true,
      "custom_build": false,
      "publish_scorer": false,
      "trust_reason": "...",
      "bounce_reason": "..."
    }}
  ]
}}

Rules:
- Probabilities must be between 0 and 1.
- Do not use everyone. Some personas should bounce.
- Be sensitive to market, job role, content job, price, and AI comfort.
- If uncertain, lower probability rather than making everyone buy.
""".strip()

def run_llm_panel(personas: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    total_calls_est = math.ceil(len(personas) / LLM_BATCH_SIZE)
    print(f"If USE_LLM_PERSONA_PANEL=True, estimated API calls: {total_calls_est}")

    if not USE_LLM_PERSONA_PANEL:
        print("Using deterministic persona model. Set USE_LLM_PERSONA_PANEL=True for Gemini/Colab panel.")
        RUN_LEDGER.append({
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "agent_name": "PersonaPanelAgent",
            "llm_calls": 0,
            "status": "deterministic_fallback",
            "personas": len(personas),
        })
        return deterministic_df.copy()

    for start in range(0, len(personas), LLM_BATCH_SIZE):
        batch = personas[start:start + LLM_BATCH_SIZE]
        expected_ids = [p["persona_id"] for p in batch]
        prompt = build_persona_panel_prompt(batch)
        try:
            raw = call_llm_text(prompt, agent_name="PersonaPanelAgent")
            parsed_rows = safe_extract_panel_json(raw, expected_persona_ids=expected_ids)

            # If model returned a list/dict but missing rows, fill missing deterministically.
            got_ids = {r.get("persona_id") for r in parsed_rows}
            for p in batch:
                if p["persona_id"] not in got_ids:
                    parsed_rows.append(deterministic_decision(p))

            # Add persona context back.
            by_id = {p["persona_id"]: p for p in batch}
            for r in parsed_rows:
                pid = r.get("persona_id")
                p = by_id.get(pid)
                if not p:
                    continue
                base = deterministic_decision(p)
                merged = {**base, **r}
                # Normalize bool/prob fields.
                for prob_key in ["try_probability", "reveal_probability", "custom_build_probability", "publish_scorer_probability"]:
                    try:
                        merged[prob_key] = max(0.0, min(1.0, float(merged.get(prob_key, base[prob_key]))))
                    except Exception:
                        merged[prob_key] = base[prob_key]
                for bool_key in ["paid_reveal", "custom_build", "publish_scorer"]:
                    val = merged.get(bool_key, base[bool_key])
                    if isinstance(val, str):
                        val = val.strip().lower() in {"true", "yes", "1"}
                    merged[bool_key] = bool(val)
                rows.append(merged)

        except Exception as e:
            print("LLM batch failed:", start, repr(e))
            RUN_LEDGER.append({
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "agent_name": "PersonaPanelAgent",
                "llm_calls": 1,
                "status": "batch_fallback_after_error",
                "batch_start": start,
                "error": repr(e),
            })
            rows.extend([deterministic_decision(p) for p in batch])

    return pd.DataFrame(rows)

panel_df = run_llm_panel(personas)

# Deterministic evals on panel output.
required_cols = ["persona_id", "market", "segment", "selected_scorer_id", "reveal_probability", "paid_reveal", "custom_build", "publish_scorer"]
checks = {
    "has_all_required_columns": all(c in panel_df.columns for c in required_cols),
    "has_all_personas": panel_df["persona_id"].nunique() == len(personas),
    "probabilities_in_range": panel_df["reveal_probability"].between(0, 1).all(),
    "not_everyone_buys": 0 < panel_df["paid_reveal"].mean() < 0.95,
    "both_markets_present": set(panel_df["market"]) == set(MARKETS),
}
print("Panel checks:", checks)
display(panel_df.head(12))
show_ledger()

If USE_LLM_PERSONA_PANEL=True, estimated API calls: 20
Panel checks: {'has_all_required_columns': True, 'has_all_personas': True, 'probabilities_in_range': np.True_, 'not_everyone_buys': np.True_, 'both_markets_present': True}


,persona_id,market,segment,job_role,content_job_to_be_done,selected_scorer_id,selected_scorer_title,price,try_probability,reveal_probability,custom_build_probability,publish_scorer_probability,paid_reveal,custom_build,publish_scorer,trust_reason,bounce_reason
0,SI_001,Singapore,sme_owner_operator,SME owner,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.90,0.70,0.10,0.05,True,False,False,Direct relevance of 'Persuasive Without Hype' ...,If the teaser output isn't sufficiently compel...
1,SI_002,Singapore,sme_owner_operator,shop owner,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.60,0.30,0.05,0.01,False,False,False,The promise of improving 'sales copy' for 'lea...,Low AI comfort leading to skepticism about the...
2,SI_003,Singapore,sme_owner_operator,clinic manager,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.95,0.80,0.30,0.10,True,False,False,High AI comfort and the alignment of 'Persuasi...,"If the improved text lacks the specific, sensi..."
3,SI_004,Singapore,sme_owner_operator,services business owner,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.90,0.85,0.60,0.10,True,True,False,Strong need for effective 'sales copy' on thei...,If the improvements aren't substantial enough ...
4,SI_005,Singapore,sme_owner_operator,e-commerce operator,sales copy,persuasive_without_hype,Persuasive Without Hype,4.99,0.50,0.20,0.05,0.01,False,False,False,"The tool promises to enhance 'sales copy', a c...",Low AI comfort combined with skepticism that A...
5,SI_006,Singapore,sme_owner_operator,SME owner,customer updates,linkedin_creator_hook,Investor-Ready,9.99,0.95,0.90,0.70,0.25,True,True,False,High AI comfort and a precise match between 'L...,Highly unlikely to bounce if the teaser provid...
6,SI_007,Singapore,sme_owner_operator,shop owner,customer updates,persuasive_without_hype,Investor-Ready,9.99,0.80,0.60,0.15,0.05,True,False,False,The tool's potential to make 'customer updates...,If the AI-improved emails lack a personal touc...
7,SI_008,Singapore,sme_owner_operator,clinic manager,customer updates,scientific_but_readable,Investor-Ready,9.99,0.60,0.25,0.05,0.01,False,False,False,The promise of 'Scientific But Readable' align...,Low AI comfort and the high stakes of patient ...
8,SI_009,Singapore,sme_owner_operator,services business owner,customer updates,scientific_but_readable,Investor-Ready,9.99,0.95,0.75,0.65,0.20,True,True,False,High AI comfort and the value of 'Scientific B...,If the AI-generated updates are too generic or...
9,SI_010,Singapore,sme_owner_operator,e-commerce operator,customer updates,persuasive_without_hype,Investor-Ready,9.99,0.85,0.70,0.35,0.10,True,False,False,The tool's ability to enhance 'customer update...,If the AI-generated updates don't align with t...


,timestamp,agent_name,llm_calls,input_tokens_est,output_tokens_est,status
0,2026-06-08T09:16:57,PersonaPanelAgent,1,1005,831,ok
1,2026-06-08T09:17:34,PersonaPanelAgent,1,1000,922,ok
2,2026-06-08T09:18:06,PersonaPanelAgent,1,992,992,ok
3,2026-06-08T09:18:44,PersonaPanelAgent,1,985,895,ok
4,2026-06-08T09:19:22,PersonaPanelAgent,1,994,1043,ok
5,2026-06-08T09:20:06,PersonaPanelAgent,1,976,700,ok
6,2026-06-08T09:20:38,PersonaPanelAgent,1,951,807,ok
7,2026-06-08T09:21:14,PersonaPanelAgent,1,996,783,ok
8,2026-06-08T09:21:58,PersonaPanelAgent,1,972,841,ok
9,2026-06-08T09:22:43,PersonaPanelAgent,1,980,946,ok


## Cell 5b — LLM panel calibration diagnostics

The persona panel is useful for qualitative market stress-testing, but raw `paid_reveal=True/False`
can be too optimistic because the model often thresholds probabilities.

This cell computes:
- expected revenue using `reveal_probability`,
- binary sampled revenue using `paid_reveal`,
- optimism / calibration flags,
- model used per batch.

In [ ]:
# LLM panel calibration diagnostics.
# Runs after `panel_df = run_llm_panel(personas)`.

def expected_economics_for_row(row: pd.Series) -> Dict[str, Any]:
    market = row["market"]
    fees = PAYMENT_FEES[market]
    card = scorer_by_id(row["selected_scorer_id"]) if "scorer_by_id" in globals() else next((c for c in SCORER_CARDS if c["scorer_id"] == row["selected_scorer_id"]), SCORER_CARDS[0])
    price = float(row.get("price", card["price_to_reveal"]))
    p = max(0.0, min(1.0, float(row.get("reveal_probability", 0.0))))

    expected_gross = price * p
    expected_payment_fee = p * (price * fees["rate"] + fees["fixed"])
    expected_api_cost = p * ESTIMATED_API_COST["preset_reveal"]
    expected_net = max(0.0, expected_gross - expected_payment_fee - expected_api_cost)
    return {
        "expected_paid_reveal": p,
        "expected_gross_revenue": expected_gross,
        "expected_payment_fee": expected_payment_fee,
        "expected_api_cost": expected_api_cost,
        "expected_net_after_fee_api": expected_net,
        "expected_platform_take": expected_net * 0.30,
        "expected_creator_take": expected_net * 0.70,
    }

expected_rows = []
for _, r in panel_df.iterrows():
    expected_rows.append({**r.to_dict(), **expected_economics_for_row(r)})

expected_sim_df = pd.DataFrame(expected_rows)

expected_summary_by_market = expected_sim_df.groupby("market").agg(
    personas=("persona_id", "count"),
    expected_paid_reveals=("expected_paid_reveal", "sum"),
    expected_conversion_rate=("expected_paid_reveal", "mean"),
    expected_gross_revenue=("expected_gross_revenue", "sum"),
    expected_platform_take=("expected_platform_take", "sum"),
    expected_creator_take=("expected_creator_take", "sum"),
).reset_index()

binary_summary_by_market = panel_df.groupby("market").agg(
    binary_paid_reveals=("paid_reveal", "sum"),
    binary_conversion_rate=("paid_reveal", "mean"),
    avg_reveal_probability=("reveal_probability", "mean"),
).reset_index()

calibration_summary = expected_summary_by_market.merge(binary_summary_by_market, on="market", how="left")
calibration_summary["binary_minus_expected_conversion"] = (
    calibration_summary["binary_conversion_rate"] - calibration_summary["expected_conversion_rate"]
)

calibration_flags = []
for _, r in calibration_summary.iterrows():
    if r["binary_conversion_rate"] > 0.50:
        calibration_flags.append(f"{r['market']}: binary conversion >50%; use expected-value mode for pitch, not binary bools.")
    if abs(r["binary_minus_expected_conversion"]) > 0.10:
        calibration_flags.append(f"{r['market']}: binary conversion differs from expected probability by >10pp.")
    if r["expected_conversion_rate"] > 0.45:
        calibration_flags.append(f"{r['market']}: expected conversion still high; present as optimistic synthetic panel, not forecast.")

model_rows = pd.DataFrame(RUN_LEDGER)
if "model_name" in model_rows.columns:
    print("Models used:")
    display(model_rows[["agent_name", "model_name", "llm_calls", "input_tokens_est", "output_tokens_est"]].head(25))
else:
    print("No model_name column in RUN_LEDGER; rerun with patched call_llm_text to log model names.")

print("Calibration flags:")
for f in calibration_flags or ["None"]:
    print("-", f)

display(calibration_summary)

# Add to packet if export cell is run later.
EXPECTED_SIM_DF = expected_sim_df
EXPECTED_SUMMARY_BY_MARKET = expected_summary_by_market
CALIBRATION_SUMMARY = calibration_summary
CALIBRATION_FLAGS = calibration_flags

## Cell 6 — Payment economics and marketplace simulation

This converts persona choices into simulated gross revenue, Stripe-style fees, API costs, platform take, and creator payouts.

In [8]:
def scorer_by_id(sid: str) -> Dict[str, Any]:
    return next((c for c in SCORER_CARDS if c["scorer_id"] == sid), SCORER_CARDS[0])

def economics_for_row(row: pd.Series) -> Dict[str, Any]:
    market = row["market"]
    fees = PAYMENT_FEES[market]
    card = scorer_by_id(row["selected_scorer_id"])
    price = float(row.get("price", card["price_to_reveal"]))
    gross = price if row["paid_reveal"] else 0.0
    fee = gross * fees["rate"] + (fees["fixed"] if gross > 0 else 0.0)
    api_cost = ESTIMATED_API_COST["preset_reveal"] if gross > 0 else 0.0
    net = max(0.0, gross - fee - api_cost)
    platform = net * 0.30
    creator = net * 0.70
    return {
        "currency": fees["currency"],
        "gross_revenue": gross,
        "payment_fee": fee,
        "api_cost": api_cost,
        "net_after_fee_api": net,
        "platform_take": platform,
        "creator_take": creator,
    }

econ_rows = []
for _, r in panel_df.iterrows():
    econ_rows.append({**r.to_dict(), **economics_for_row(r)})

sim_df = pd.DataFrame(econ_rows)

summary_by_market = sim_df.groupby("market").agg(
    personas=("persona_id", "count"),
    paid_reveals=("paid_reveal", "sum"),
    conversion_rate=("paid_reveal", "mean"),
    custom_builds=("custom_build", "sum"),
    publish_scorers=("publish_scorer", "sum"),
    gross_revenue=("gross_revenue", "sum"),
    payment_fees=("payment_fee", "sum"),
    api_cost=("api_cost", "sum"),
    platform_take=("platform_take", "sum"),
    creator_take=("creator_take", "sum"),
).reset_index()

summary_by_segment = sim_df.groupby(["market", "segment"]).agg(
    personas=("persona_id", "count"),
    avg_reveal_prob=("reveal_probability", "mean"),
    paid_reveals=("paid_reveal", "sum"),
    conversion_rate=("paid_reveal", "mean"),
    custom_builds=("custom_build", "sum"),
    publish_scorers=("publish_scorer", "sum"),
    gross_revenue=("gross_revenue", "sum"),
).reset_index()

summary_by_scorer = sim_df.groupby(["market", "selected_scorer_id"]).agg(
    personas=("persona_id", "count"),
    avg_reveal_prob=("reveal_probability", "mean"),
    paid_reveals=("paid_reveal", "sum"),
    gross_revenue=("gross_revenue", "sum"),
    platform_take=("platform_take", "sum"),
    creator_take=("creator_take", "sum"),
).reset_index()

display(summary_by_market)
display(summary_by_segment.sort_values(["market", "conversion_rate"], ascending=[True, False]))
display(summary_by_scorer.sort_values(["market", "gross_revenue"], ascending=[True, False]))

,market,personas,paid_reveals,conversion_rate,custom_builds,publish_scorers,gross_revenue,payment_fees,api_cost,platform_take,creator_take
0,Singapore,100,59,0.59,26,16,351.41,41.44794,20.65,86.793618,202.518442
1,United States,100,61,0.61,35,20,364.39,28.86731,21.35,94.251807,219.920883


,market,segment,personas,avg_reveal_prob,paid_reveals,conversion_rate,custom_builds,publish_scorers,gross_revenue
2,Singapore,marketing_growth_lead,14,0.664286,11,0.785714,3,0,56.89
3,Singapore,researcher_technical_writer,8,0.625000,6,0.750000,3,0,29.94
7,Singapore,startup_founder_operator,14,0.682143,10,0.714286,6,3,74.90
6,Singapore,sme_owner_operator,16,0.596875,11,0.687500,3,0,74.89
4,Singapore,sales_bd_customer_success,12,0.550000,8,0.666667,2,0,39.92
0,Singapore,agency_freelancer,10,0.550000,5,0.500000,6,5,24.95
1,Singapore,creator_coach_consultant,13,0.473077,5,0.384615,3,8,34.95
8,Singapore,student_job_seeker,7,0.307143,2,0.285714,0,0,9.98
5,Singapore,skeptical_control,6,0.350000,1,0.166667,0,0,4.99
11,United States,marketing_growth_lead,14,0.728571,12,0.857143,7,0,63.88


,market,selected_scorer_id,personas,avg_reveal_prob,paid_reveals,gross_revenue,platform_take,creator_take
2,Singapore,persuasive_without_hype,67,0.531343,37,198.63,48.127974,112.298606
1,Singapore,linkedin_creator_hook,15,0.616667,9,55.91,13.907718,32.451342
0,Singapore,investor_ready,6,0.708333,5,49.95,13.200510,30.801190
3,Singapore,scientific_but_readable,12,0.566667,8,46.92,11.557416,26.967304
6,United States,persuasive_without_hype,68,0.541176,40,219.60,56.169480,131.062120
5,United States,linkedin_creator_hook,16,0.653125,12,74.88,19.472544,45.435936
4,United States,investor_ready,6,0.700000,5,49.95,13.575435,31.676015
7,United States,scientific_but_readable,10,0.470000,4,19.96,5.034348,11.746812


## Cell 7 — Price sweep

This tests which reveal price might maximize revenue and conversion for each market.

The deterministic decision function is used for price sweeps even if LLM panel is enabled, because asking the LLM for every price/persona would explode call counts.

In [9]:
price_rows = []
for market in MARKETS:
    market_personas = [p for p in personas if p["market"] == market]
    for price in PRICE_POINTS:
        decisions = [deterministic_decision(p, price_override=price) for p in market_personas]
        df = pd.DataFrame(decisions)
        # Override selected price for econ
        df["price"] = price
        econ = []
        for _, r in df.iterrows():
            econ.append({**r.to_dict(), **economics_for_row(r)})
        edf = pd.DataFrame(econ)
        price_rows.append({
            "market": market,
            "price": price,
            "personas": len(edf),
            "conversion_rate": edf["paid_reveal"].mean(),
            "paid_reveals": int(edf["paid_reveal"].sum()),
            "gross_revenue": edf["gross_revenue"].sum(),
            "platform_take": edf["platform_take"].sum(),
            "creator_take": edf["creator_take"].sum(),
            "net_after_fee_api": edf["net_after_fee_api"].sum(),
        })

price_sweep_df = pd.DataFrame(price_rows)

display(price_sweep_df.sort_values(["market", "platform_take"], ascending=[True, False]))

best_prices = price_sweep_df.sort_values("platform_take", ascending=False).groupby("market").head(1)
display(Markdown("## Best demo price per market by platform take"))
display(best_prices)

,market,price,personas,conversion_rate,paid_reveals,gross_revenue,platform_take,creator_take,net_after_fee_api
3,Singapore,7.99,100,0.32,32,255.68,65.936064,153.850816,219.78688
4,Singapore,9.99,100,0.21,21,209.79,55.442142,129.364998,184.80714
2,Singapore,4.99,100,0.36,36,179.64,42.879672,100.052568,142.93224
5,Singapore,19.99,100,0.05,5,99.95,27.690510,64.611190,92.30170
1,Singapore,2.99,100,0.44,44,131.56,26.906088,62.780872,89.68696
0,Singapore,1.99,100,0.49,49,97.51,15.763398,36.781262,52.54466
10,United States,9.99,100,0.25,25,249.75,67.877175,158.380075,226.25725
8,United States,4.99,100,0.52,52,259.48,65.446524,152.708556,218.15508
9,United States,7.99,100,0.21,21,167.79,44.782227,104.491863,149.27409
11,United States,19.99,100,0.05,5,99.95,28.140435,65.661015,93.80145


## Best demo price per market by platform take

,market,price,personas,conversion_rate,paid_reveals,gross_revenue,platform_take,creator_take,net_after_fee_api
10,United States,9.99,100,0.25,25,249.75,67.877175,158.380075,226.25725
3,Singapore,7.99,100,0.32,32,255.68,65.936064,153.850816,219.78688


## Cell 8 — Failure diagnostics

This cell flags market-simulation failure modes that could scupper the demo.

In [10]:
diagnostics = {}

for market in MARKETS:
    mdf = sim_df[sim_df["market"] == market]
    diagnostics[market] = {
        "conversion_rate": float(mdf["paid_reveal"].mean()),
        "custom_build_rate": float(mdf["custom_build"].mean()),
        "publish_scorer_rate": float(mdf["publish_scorer"].mean()),
        "dominant_segment_share": float(mdf["segment"].value_counts(normalize=True).max()),
        "dominant_scorer_share": float(mdf["selected_scorer_id"].value_counts(normalize=True).max()),
        "avg_reveal_probability": float(mdf["reveal_probability"].mean()),
    }

diag_df = pd.DataFrame([{"market": k, **v} for k, v in diagnostics.items()])

failure_flags = []
for _, r in diag_df.iterrows():
    if r["conversion_rate"] <= 0.05:
        failure_flags.append(f"{r['market']}: conversion too low for pay-to-reveal story")
    if r["conversion_rate"] >= 0.80:
        failure_flags.append(f"{r['market']}: conversion implausibly high; simulator too optimistic")
    if r["dominant_scorer_share"] >= 0.80:
        failure_flags.append(f"{r['market']}: one scorer dominates; marketplace variety story weak")
    if r["publish_scorer_rate"] <= 0.03:
        failure_flags.append(f"{r['market']}: creator marketplace supply weak")
    if r["custom_build_rate"] <= 0.05:
        failure_flags.append(f"{r['market']}: custom evaluator build demand weak")

print("Failure flags:")
for f in failure_flags or ["None"]:
    print("-", f)

display(diag_df)
display(pd.DataFrame({"failure_flags": failure_flags or ["None"]}))

Failure flags:
- None


,market,conversion_rate,custom_build_rate,publish_scorer_rate,dominant_segment_share,dominant_scorer_share,avg_reveal_probability
0,Singapore,0.59,0.26,0.16,0.16,0.67,0.5590
1,United States,0.61,0.35,0.20,0.16,0.68,0.5615


,failure_flags
0,None


## Cell 9 — Export artifacts

Send back:

```text
/content/evalweaver_persona_sim_v2_outputs/persona_market_sim_v2_packet.json
```

In [11]:
packet = {
    "run_name": "evalweaver_persona_market_sim_v2",
    "markets": MARKETS,
    "personas_per_market": PERSONAS_PER_MARKET,
    "use_llm_persona_panel": USE_LLM_PERSONA_PANEL,
    "llm_batch_size": LLM_BATCH_SIZE,
    "estimated_llm_calls_if_enabled": math.ceil(len(personas) / LLM_BATCH_SIZE),
    "segment_frame": SEGMENT_FRAME,
    "scorer_cards": SCORER_CARDS,
    "personas": personas_df.to_dict(orient="records"),
    "panel_results": panel_df.to_dict(orient="records"),
    "simulation": sim_df.to_dict(orient="records"),
    "summary_by_market": summary_by_market.to_dict(orient="records"),
    "summary_by_segment": summary_by_segment.to_dict(orient="records"),
    "summary_by_scorer": summary_by_scorer.to_dict(orient="records"),
    "price_sweep": price_sweep_df.to_dict(orient="records"),
    "diagnostics": diagnostics,
    "failure_flags": failure_flags,
    "run_ledger": RUN_LEDGER,
    "expected_summary_by_market": EXPECTED_SUMMARY_BY_MARKET.to_dict(orient="records") if "EXPECTED_SUMMARY_BY_MARKET" in globals() else [],
    "calibration_summary": CALIBRATION_SUMMARY.to_dict(orient="records") if "CALIBRATION_SUMMARY" in globals() else [],
    "calibration_flags": CALIBRATION_FLAGS if "CALIBRATION_FLAGS" in globals() else [],
}

packet_path = OUTDIR / "persona_market_sim_v2_packet.json"
report_path = OUTDIR / "persona_market_sim_v2_report.md"

packet_path.write_text(json.dumps(packet, indent=2), encoding="utf-8")

personas_df.to_csv(OUTDIR / "personas_v2.csv", index=False)
panel_df.to_csv(OUTDIR / "persona_panel_results_v2.csv", index=False)
sim_df.to_csv(OUTDIR / "persona_card_simulation_v2.csv", index=False)
summary_by_market.to_csv(OUTDIR / "summary_by_market_v2.csv", index=False)
summary_by_segment.to_csv(OUTDIR / "summary_by_segment_v2.csv", index=False)
summary_by_scorer.to_csv(OUTDIR / "summary_by_scorer_v2.csv", index=False)
price_sweep_df.to_csv(OUTDIR / "price_sweep_v2.csv", index=False)
if "EXPECTED_SUMMARY_BY_MARKET" in globals():
    EXPECTED_SUMMARY_BY_MARKET.to_csv(OUTDIR / "expected_summary_by_market_v2.csv", index=False)
if "CALIBRATION_SUMMARY" in globals():
    CALIBRATION_SUMMARY.to_csv(OUTDIR / "calibration_summary_v2.csv", index=False)

report = []
report.append("# EvalWeaver Persona Market Simulator v2 Report\n")
report.append(f"Markets: {', '.join(MARKETS)}\n")
report.append(f"Personas per market: {PERSONAS_PER_MARKET}\n")
report.append(f"LLM panel enabled: {USE_LLM_PERSONA_PANEL}\n")
report.append(f"Estimated LLM calls if enabled: {math.ceil(len(personas) / LLM_BATCH_SIZE)}\n")
report.append("## Summary by market\n")
report.append(summary_by_market.to_markdown(index=False))
report.append("\n## Best prices by platform take\n")
report.append(best_prices.to_markdown(index=False))
report.append("\n## Summary by scorer\n")
report.append(summary_by_scorer.to_markdown(index=False))
report.append("\n## Failure flags\n")
report.extend([f"- {x}" for x in (failure_flags or ["None"])])
report_path.write_text("\n".join(report), encoding="utf-8")

print("✅ Exported:")
print(packet_path)
print(report_path)
print(OUTDIR)
display(summary_by_market)

✅ Exported:
/content/evalweaver_persona_sim_v2_outputs/persona_market_sim_v2_packet.json
/content/evalweaver_persona_sim_v2_outputs/persona_market_sim_v2_report.md
/content/evalweaver_persona_sim_v2_outputs


,market,personas,paid_reveals,conversion_rate,custom_builds,publish_scorers,gross_revenue,payment_fees,api_cost,platform_take,creator_take
0,Singapore,100,59,0.59,26,16,351.41,41.44794,20.65,86.793618,202.518442
1,United States,100,61,0.61,35,20,364.39,28.86731,21.35,94.251807,219.920883
